# Train on a free GPU (Colab / Kaggle)

Why: training on the Mac (MPS) was slow and silently stalled for two epochs. On a T4 GPU this takes about 30-45 min per model.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
Outputs (tables, figures, best model) are zipped at the end so you can download them and commit the small files to GitHub.

In [1]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
# 1. Get the code (the repo must contain the latest src/, push from your Mac first)
!rm -rf Hate-Speech-Detection
!git clone https://github.com/aadya197/hate-speech-detection Hate-Speech-Detection
%cd Hate-Speech-Detection
!pip install -q "transformers>=4.45" datasets "accelerate>=1.0" scikit-learn seaborn matplotlib

Cloning into 'Hate-Speech-Detection'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 65 (delta 15), reused 59 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 464.78 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Hate-Speech-Detection


In [3]:
# 2. Build the processed CSVs (downloads HateXplain, needs internet; about 1 minute)
!python -m src.data
!ls -la data/processed


plain_text/hatexplain-train.parquet: downloading bytes:   0% 0.00/1.68M [00:00<?, ?B/s]
plain_text/hatexplain-train.parquet: downloading bytes: 100% 1.67M/1.67M [00:00<00:00, 2.44MB/s,  165kB/s  ]
plain_text/hatexplain-train.parquet: reconstructing file: 100% 1.68M/1.68M [00:00<00:00, 2.45MB/s,  166kB/s  ]
hatexplain-validation.parquet: 100% 212k/212k [00:00<00:00, 12.0MB/s]
hatexplain-test.parquet: 100% 212k/212k [00:00<00:00, 48.3MB/s]
Generating train split: 15383 examples [00:00, 264760.10 examples/s]
Generating validation split: 1922 examples [00:00, 235405.24 examples/s]
Generating test split: 1924 examples [00:00, 247510.76 examples/s]
train: (15383, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_train.csv
validation: (1922, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_validation.csv
test: (1924, 6) -> /content/Hate-Speech-Detection/data/processed/hatexplain_test.csv
total 5192
drwxr-xr-x 2 root root    4096 Nov  9 10:45 .
drwxr-xr-x 3 root roo

In [4]:
# 3. Quick sanity check: 1 tiny epoch, makes sure the whole pipeline works on this GPU
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --smoke --train_bs 16 --grad_accum 1 --fp16

usage: train_xlmr.py [-h] [--model_name MODEL_NAME] [--epochs EPOCHS]
                     [--lr LR] [--train_bs TRAIN_BS] [--grad_accum GRAD_ACCUM]
                     [--resume] [--smoke] [--tag TAG]
train_xlmr.py: error: unrecognized arguments: --fp16


## Model A: BERT (main model)
`bert-base-uncased` is the standard model for HateXplain and is English-only, which suits these posts.
Watch the log: `eval_macro_f1` should **change and grow** every epoch. If it prints the WARNING about identical metrics, stop.

In [5]:
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --epochs 4 --train_bs 16 --grad_accum 1 --fp16

usage: train_xlmr.py [-h] [--model_name MODEL_NAME] [--epochs EPOCHS]
                     [--lr LR] [--train_bs TRAIN_BS] [--grad_accum GRAD_ACCUM]
                     [--resume] [--smoke] [--tag TAG]
train_xlmr.py: error: unrecognized arguments: --fp16


## Model B: XLM-RoBERTa again, now on GPU (for the comparison table)
Optional but recommended: the Mac run only really trained one epoch, so its 0.56 macro-F1 is not a fair number.
XLM-R is sensitive to the learning rate; 2e-5 with batch size 16 is a safe choice.

In [6]:
!python -m src.train_xlmr --model_name FacebookAI/xlm-roberta-base --tag xlmr_gpu --epochs 4 --train_bs 16 --grad_accum 1 --fp16

usage: train_xlmr.py [-h] [--model_name MODEL_NAME] [--epochs EPOCHS]
                     [--lr LR] [--train_bs TRAIN_BS] [--grad_accum GRAD_ACCUM]
                     [--resume] [--smoke] [--tag TAG]
train_xlmr.py: error: unrecognized arguments: --fp16


In [7]:
# 4. Show the results
import pandas as pd
for tag in ["bert", "xlmr_gpu"]:
    try:
        print(tag); display(pd.read_csv(f"results/tables/{tag}_metrics.csv")); display(pd.read_csv(f"results/tables/{tag}_per_class_test.csv"))
    except FileNotFoundError:
        print(tag, "not run yet")

bert
bert not run yet
xlmr_gpu
xlmr_gpu not run yet


In [8]:
# 5. Zip everything you need to keep and download it
!zip -r results_gpu.zip results/tables results/figures models/bert_best models/xlmr_gpu_best -x "*checkpoint*"
from google.colab import files   # on Kaggle: the zip appears under /kaggle/working instead
files.download("results_gpu.zip")

	zip warning: name not matched: models/bert_best
	zip warning: name not matched: models/xlmr_gpu_best
  adding: results/tables/ (stored 0%)
  adding: results/tables/shap_rationale_alignment.csv (deflated 65%)
  adding: results/tables/xlmr_metrics.csv (deflated 35%)
  adding: results/tables/baseline_metrics.csv (deflated 29%)
  adding: results/tables/shap_faithfulness.csv (deflated 31%)
  adding: results/tables/preliminary_shap_rationale_coverage.csv (deflated 55%)
  adding: results/tables/xlmr_training_log.csv (deflated 63%)
  adding: results/tables/xlmr_smoke_metrics.csv (deflated 37%)
  adding: results/tables/xlmr_smoke_training_log.csv (deflated 51%)
  adding: results/tables/xlmr_per_class_test.csv (deflated 29%)
  adding: results/tables/xlmr_smoke_per_class_test.csv (deflated 25%)
  adding: results/tables/shap_faithfulness_multi_example.csv (deflated 47%)
  adding: results/figures/ (stored 0%)
  adding: results/figures/xlmr_confusion_matrix_test.png (deflated 12%)
  adding: results

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After downloading, unzip into your project folder on the Mac. `models/` is git-ignored (large), but `results/tables` and `results/figures` should be committed.